In [1]:
import os
import gzip
import struct
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam


In [2]:
def convolve(sequence, weights):
    convolution = np.zeros(len(sequence) - len(weights) + 1)
    for i in range(convolution.shape[0]):
        convolution[i] = np.sum(
            np.array(weights) * np.array(sequence[i : i + len(weights)])
        )
    return convolution

In [3]:
s = [2, 3, 5, 7, 11]
w = [-1, 1]

In [4]:
convolve(s,w)

array([1., 2., 2., 4.])

In [5]:
def convolve2d(image, kernel):
    image = np.array(image)
    kernel = np.array(kernel)
    m, n = image.shape[0], kernel.shape[0]
    size = m - n + 1                      # posiciones donde cabe el kernel
    result = np.zeros((size, size))
    for i in range(size):                 # de arriba a abajo
        for j in range(size):             # de izquierda a derecha
            block = image[i:i + n, j:j + n]           # bloque n×n de la imagen
            result[i, j] = np.sum(block * kernel)     # multiplica y suma
    return result

s = [[2, 1, 3],
     [4, 0, 2],
     [1, 5, 6]]

w = [[-1, -2],
     [ 1,  2]]

print(convolve2d(s, w))

[[ 0. -3.]
 [ 7. 13.]]


## Adam model

In [9]:
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import Conv2D, Flatten, Dense, AvgPool2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import numpy as np

def load_data():
    (features_train, target_train), (features_test, target_test) = fashion_mnist.load_data()
    features_train = features_train.reshape(-1, 28, 28, 1) / 255.0
    features_test = features_test.reshape(-1, 28, 28, 1) / 255.0
    return (features_train, target_train), (features_test, target_test)

def create_model(input_shape):
    model = Sequential()
    
    model.add(
        Conv2D(
            filters=6, kernel_size=(5, 5), padding='same', activation='relu', input_shape=input_shape
        )
    )

    model.add(AvgPool2D(pool_size=(2, 2)))

    model.add(
        Conv2D(
            filters=16, kernel_size=(5, 5), padding='valid', activation='relu'
        )
    )

    model.add(AvgPool2D(pool_size=(2, 2)))

    model.add(Flatten())

    model.add(Dense(120, activation='relu'))
    model.add(Dense(84, activation='relu'))
    model.add(Dense(10, activation='softmax'))

    optimizer = Adam()

    model.compile(
        optimizer= optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['acc'],
    )

    return model

def train_model(
        model,
        train_data,
        test_data,
        batch_size=32,
        epochs=5,
        steps_per_epoch=None,
        validation_steps=None,
):

    features_train, target_train = train_data
    features_test, target_test = test_data

    model.fit(
        features_train,
        target_train,
        validation_data=(features_test, target_test),
        batch_size=batch_size,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=2,
        shuffle=True,
    )

    return model

train_data, test_data = load_data()
input_shape = (28, 28, 1)
model = create_model(input_shape)

model_fitted = train_model(model,
                           train_data,
                           test_data,
                           epochs=2
                           )

/Users/carloscarranza/miniconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/2
1875/1875 - 11s - 6ms/step - acc: 0.7889 - loss: 0.5709 - val_acc: 0.8292 - val_loss: 0.4692
Epoch 2/2
1875/1875 - 10s - 5ms/step - acc: 0.8579 - loss: 0.3897 - val_acc: 0.8539 - val_loss: 0.3968
